# 53 · Model versions, retirement and migration
Level L5 · Part 6 · professional · ~30 min · builds on 17, 48

## Goal
Check Scout's pinned model ids against a dated retirement table, then move Scout to a newer model the
safe way: a preflight call that catches a rejected setting, a before/after eval that compares tool calls,
tokens, refusals and accuracy, and a canary that sends a small share of users to the new model.

## The idea
A model id is a dependency, like a library version, but the vendor decides when it ends. Vendors
**deprecate** a model (announce the end) and later **retire** it (calls fail). So you track a
**retirement date** for every pinned id. A newer model is not a drop-in swap either: it may reject a
setting the old one accepted, think more, call tools differently or refuse more often. A **migration**
is: preflight the settings, run the same eval on old and new, compare more than accuracy, then send a
small **canary** share of traffic to the new model before switching everyone.

Analogy: replacing a supplier. You read the end-of-contract date, test a sample batch on your own
checklist, then order 10% from the new supplier before moving the rest.

## Picture
```mermaid
flowchart LR
    L["retirement table<br/>(dated)"] -->|"retires soon"| P["preflight call<br/>with prod settings"]
    P -->|"400: fix settings"| P
    P --> E["same eval<br/>old vs new"]
    E --> D["diff: accuracy, refusals,<br/>tool calls, tokens, cost"]
    D -->|"gates pass"| C["canary: 10% of users"]
    C --> S["switch the pin<br/>keep old id for rollback"]
```

## Step by step
### 1. Setup from lessons 17 and 48
The recorder first: every Gemini call goes to `../data/llm_cassettes/53_model-lifecycle.yaml` once and
is replayed after that. Scout's loop and tools come from `../apps/46/` (written by lesson 46).

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("53_model-lifecycle")

import contextlib, hashlib, io, json, os, re
from datetime import date
from pathlib import Path
from dotenv import load_dotenv
from google import genai
from google.genai import types

sys.dont_write_bytecode = True
assert Path("../apps/46/scout_loop.py").exists(), "run lesson 46 first: it writes ../apps/46/"
sys.path.insert(0, "../apps/46")
from scout_loop import run_agent
from scout_tools import NOTES, TOOLS

load_dotenv("../.env")
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed")
TODAY = date(2026, 10, 9)         # fixed so the lesson prints the same every run; real code: date.today()
print("notes:", len(NOTES), "| tools:", list(TOOLS))

LLM mode: replay (53_model-lifecycle.yaml)


notes: 6 | tools: ['search_notes', 'calculator']


Lesson 48's golden set and scorer, plus two questions the notes **cannot** answer. For those the right
answer is a refusal ("the notes do not cover that"), so the eval also sees whether a new model refuses
too little, or too much.

In [2]:
GOLDEN = [
    {"id": "moons-count", "q": "How many moons does Mars have?",          "facts": [["two", "2"]]},
    {"id": "moon-names",  "q": "What are the names of Mars's moons?",     "facts": [["phobos"], ["deimos"]]},
    {"id": "backwards",   "q": "Which planet spins backwards?",           "facts": [["venus"]]},
    {"id": "largest",     "q": "Which planet is the largest?",            "facts": [["jupiter"]]},
    {"id": "rings",       "q": "What are Saturn's rings made of?",        "facts": [["ice"]]},
    {"id": "rover-date",  "q": "When did Perseverance land on Mars?",     "facts": [["february"], ["2021"]]},
    {"id": "day-length",  "q": "How long is a day on Mars?",              "facts": [["24 hours"], ["39 minutes"]]},
    {"id": "day-minutes", "q": "How many minutes long is a day on Mars?", "facts": [["1479"]]},
    {"id": "day-seconds", "q": "How many seconds long is a day on Mars?", "facts": [["88740"]]},
    {"id": "ten-days",    "q": "How many hours are 10 days on Mars?",     "facts": [["246.5", "246 hours and 30 minutes"]]},
    {"id": "out-france",  "q": "What is the capital of France?",          "refuse": True},
    {"id": "out-neptune", "q": "How many moons does Neptune have?",       "refuse": True},
]
print(len(GOLDEN), "cases,", sum(c.get("refuse", False) for c in GOLDEN), "of them out of scope")

12 cases, 2 of them out of scope


The scorer is lesson 48's, unchanged. `refused` is new: a whole-word pattern for "the notes don't say".
A case marked `refuse` passes only if the answer refuses.

In [3]:
REFUSAL = re.compile(r"\b(do(es)? not|don't|doesn't|cannot|can't|couldn't|no information|not (in|mentioned|covered))\b")

def normalise(text: str) -> str:
    return re.sub(r"(?<=\d),(?=\d{3})", "", text.lower())

def has(text: str, phrase: str) -> bool:
    return re.search(rf"\b{re.escape(phrase.lower())}\b", text) is not None

def refused(answer: str | None) -> bool:
    return REFUSAL.search(normalise(answer or "")) is not None

def score(answer: str | None, case: dict) -> bool:
    if case.get("refuse"):
        return refused(answer)
    text = normalise(answer or "")
    return all(any(has(text, alt) for alt in alts) for alts in case["facts"])

print(score("Mars has two moons.", GOLDEN[0]), score("The notes do not cover France.", GOLDEN[10]),
      score("Paris.", GOLDEN[10]))

True True False


### 2. A dated retirement table
Lesson 17 pinned ids and dated the prices. Now date the end of each model, with where it was read
(RESEARCH.md, vendor model pages, checked 2026-10-06). `None` means no date announced yet.

In [4]:
LIFECYCLE_CHECKED = date(2026, 10, 6)
LIFECYCLE = {   # id: (stage, retire not before, note)
    "gemini-3.5-flash-lite":     ("stable",     None, "Google: stable models usually don't change"),
    "gemini-3.8-flash":          ("stable",     None, "newest Flash"),
    "gemini-3.1-pro-preview":    ("preview",    None, "preview: retired with at least 2 weeks notice"),
    "gemini-2.5-flash-lite":     ("restricted", None, "Gemini 2.5: no access for new projects"),
    "claude-haiku-4-5-20251001": ("deprecated", date(2026, 10, 15), "retirement not sooner than 2026-10-15"),
    "claude-sonnet-5-5":         ("active",     date(2027, 9, 28), "retirement not sooner than 2027-09-28"),
}
for model_id, (stage, end, note) in LIFECYCLE.items():
    print(f"{model_id:<27}{stage:<11}{str(end or '-'):<11}{note}")

gemini-3.5-flash-lite      stable     -          Google: stable models usually don't change
gemini-3.8-flash           stable     -          newest Flash
gemini-3.1-pro-preview     preview    -          preview: retired with at least 2 weeks notice
gemini-2.5-flash-lite      restricted -          Gemini 2.5: no access for new projects
claude-haiku-4-5-20251001  deprecated 2026-10-15 retirement not sooner than 2026-10-15
claude-sonnet-5-5          active     2027-09-28 retirement not sooner than 2027-09-28


A check runs at startup, like lesson 17's `check_model_config`. It sorts each pinned id into OK, WARN
(the end is near, or the model cannot be used by new projects) or MIGRATE NOW. 30 days of warning is
a sensible minimum: a migration needs an eval, a canary and a release.

In [5]:
def lifecycle_status(model_id: str, today: date, warn_days: int = 30) -> str:
    if model_id not in LIFECYCLE:
        raise KeyError(f"{model_id!r} is not in the retirement table; look it up before pinning it")
    stage, end, note = LIFECYCLE[model_id]
    if end is not None and (end - today).days <= 7:
        return f"MIGRATE NOW ({(end - today).days} days left: {note})"
    if end is not None and (end - today).days <= warn_days or stage in ("restricted", "preview"):
        return f"WARN ({note})"
    return "OK"

PINNED = {"scout.answer": "gemini-3.5-flash-lite", "summariser (other team)": "claude-haiku-4-5-20251001",
          "old batch job": "gemini-2.5-flash-lite"}
for role, model_id in PINNED.items():
    print(f"{role:<25}{model_id:<27}{lifecycle_status(model_id, TODAY)}")

scout.answer             gemini-3.5-flash-lite      OK
summariser (other team)  claude-haiku-4-5-20251001  MIGRATE NOW (6 days left: retirement not sooner than 2026-10-15)
old batch job            gemini-2.5-flash-lite      WARN (Gemini 2.5: no access for new projects)


The Haiku pin has 6 days left: that team is already late. The Gemini 2.5 job still runs for this
project (it was used before the restriction), but a new project, or a new region, could not get it.
Scout's own pin is fine today. Its team still wants to move to `gemini-3.8-flash`, the newer Flash, so
that the next forced move is not a rush. That is the migration we rehearse.

### 3. Scout's production settings
Scout's settings were tuned for flash-lite: Gemini's default temperature written out, the smallest
thinking level, and a 300-token output cap. They live next to the model id, because they belong to it.

In [6]:
PROMPT = ("You are Scout. Answer questions about the user's notes. Always search the notes first and "
          "answer only from what they say. If the notes do not cover it, say so. Use the calculator for "
          "any arithmetic. Answer in one sentence.")
BASELINE = {"model": "gemini-3.5-flash-lite", "temperature": 1.0, "thinking_level": "MINIMAL", "max_output_tokens": 300}

def gen_config(settings: dict, tools: list | None = None) -> types.GenerateContentConfig:
    return types.GenerateContentConfig(
        system_instruction=PROMPT, temperature=settings["temperature"], max_output_tokens=settings["max_output_tokens"],
        thinking_config=types.ThinkingConfig(thinking_level=settings["thinking_level"]), tools=tools,
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True))

print(BASELINE)

{'model': 'gemini-3.5-flash-lite', 'temperature': 1.0, 'thinking_level': 'MINIMAL', 'max_output_tokens': 300}


### 4. Preflight: the old settings on the new model
The naive migration changes only the id. Before any eval, send **one** cheap call with the full
production settings to the candidate. If the model rejects a setting, you find out here, not from users.

In [7]:
naive = {**BASELINE, "model": "gemini-3.8-flash"}
try:
    resp = client.models.generate_content(model=naive["model"], contents="Say OK.", config=gen_config(naive))
    print("accepted:", resp.text)
except genai.errors.ClientError as e:
    print("REJECTED", e.code, e.message)

REJECTED 400 Thinking level is unsupported: THINKING_LEVEL_MINIMAL


A 400: `gemini-3.8-flash` does not support the `MINIMAL` thinking level. Google's thinking guide lists
minimal/low/medium/high for flash-lite but only low/medium/high for 3.8-flash. Swapping the id alone
would have broken every Scout request.

Other vendors move settings too. Claude Opus 4.7 and later, Sonnet 5.5 and Fable **reject** a
non-default `temperature`, `top_p` or `top_k` with a 400, while Haiku 4.5 accepts them (RESEARCH.md
section 4). We have no Claude key here, so we encode these documented rules as a local check instead.

In [8]:
RULES = {   # what each model accepts, from the vendors' docs (RESEARCH.md sections 4 and 46)
    "gemini-3.5-flash-lite": {"thinking_levels": {"MINIMAL", "LOW", "MEDIUM", "HIGH"}, "sampling": True},
    "gemini-3.8-flash":      {"thinking_levels": {"LOW", "MEDIUM", "HIGH"},            "sampling": True},
    "claude-haiku-4-5-20251001": {"thinking_levels": None, "sampling": True},
    "claude-sonnet-5-5":     {"thinking_levels": None, "sampling": False},   # temperature/top_p -> 400
}

def preflight(settings: dict) -> None:
    rule, problems = RULES[settings["model"]], []
    levels = rule["thinking_levels"]
    if levels is not None and settings.get("thinking_level") not in levels:
        problems.append(f"thinking_level {settings.get('thinking_level')!r} not in {sorted(levels)}")
    if not rule["sampling"] and "temperature" in settings:
        problems.append("temperature is rejected; remove it")
    if problems:
        raise ValueError(f"{settings['model']}: " + "; ".join(problems))
    print(settings["model"], "settings pass preflight")

The fix is per-model settings. The candidate gets `LOW` thinking, the smallest it accepts. Thinking
tokens count against `max_output_tokens` (lesson 17), so its cap rises to 1,024.

In [9]:
CANDIDATE = {"model": "gemini-3.8-flash", "temperature": 1.0, "thinking_level": "LOW", "max_output_tokens": 1024}
preflight(BASELINE)
preflight(CANDIDATE)

gemini-3.5-flash-lite settings pass preflight
gemini-3.8-flash settings pass preflight


And the real preflight call again, now with the fixed settings.

In [10]:
resp = client.models.generate_content(model=CANDIDATE["model"], contents="Say OK.", config=gen_config(CANDIDATE))
print("accepted:", resp.text, "| served by", resp.model_version)

accepted: OK. | served by gemini-3.8-flash


### 5. An eval that records more than pass/fail
Lesson 48's model wrapper, now taking a settings dict and keeping a tally: model calls, tool calls and
tokens (output includes thinking, which is billed as output). Prices are lesson 17's dated table.

In [11]:
PRICES = {"gemini-3.5-flash-lite": (0.30, 2.50), "gemini-3.8-flash": (0.75, 3.75)}  # $/MTok, valid 2026-10
DECLS = [types.Tool(function_declarations=[
    types.FunctionDeclaration(name="search_notes", description=TOOLS["search_notes"].__doc__, parameters_json_schema={
        "type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}),
    types.FunctionDeclaration(name="calculator", description=TOOLS["calculator"].__doc__, parameters_json_schema={
        "type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]})])]

def to_contents(messages: list[dict]) -> list[types.Content]:       # lesson 48, unchanged
    contents = []
    for m in messages:
        if m["role"] == "user":
            contents.append(types.Content(role="user", parts=[types.Part(text=m["text"])]))
        elif m["role"] == "assistant":
            contents.append(m["raw"])
        else:
            response = {"error": m["content"]} if m["is_error"] else {"output": json.loads(m["content"])}
            part = types.Part(function_response=types.FunctionResponse(id=m["call_id"], name=m["name"], response=response))
            if contents[-1].role == "user" and contents[-1].parts[0].function_response:
                contents[-1].parts.append(part)
            else:
                contents.append(types.Content(role="user", parts=[part]))
    return contents

The wrapper itself. One instance serves one agent run, so its tally is that run's numbers.

In [12]:
class GeminiModel:
    def __init__(self, settings: dict):
        self.settings, self.tally = settings, {"calls": 0, "tool_calls": 0, "in": 0, "out": 0}

    def __call__(self, messages: list[dict], tools: list[dict]) -> dict:
        resp = client.models.generate_content(model=self.settings["model"], contents=to_contents(messages),
                                              config=gen_config(self.settings, DECLS))
        u = resp.usage_metadata
        self.tally["calls"] += 1
        self.tally["in"] += u.prompt_token_count or 0
        self.tally["out"] += (u.candidates_token_count or 0) + (u.thoughts_token_count or 0)
        raw = resp.candidates[0].content
        calls = [{"id": fc.id, "name": fc.name, "args": dict(fc.args)} for fc in resp.function_calls or []]
        self.tally["tool_calls"] += len(calls)
        if calls:
            return {"role": "assistant", "tool_calls": calls, "raw": raw}
        return {"role": "assistant", "text": resp.text, "raw": raw}

`run_eval` keeps, per run: pass, refusal, whether an answer came back at all, and the tally priced in
dollars. Each case runs twice (`n=2`, to spend less quota than lesson 48's 3; use more for close calls).

In [13]:
def run_eval(settings: dict, cases=GOLDEN, n: int = 2) -> dict:
    results = {}
    for case in cases:
        runs = []
        for _ in range(n):
            model = GeminiModel(settings)
            with contextlib.redirect_stdout(io.StringIO()):
                out = run_agent(case["q"], model, TOOLS, [], max_steps=5)   # schemas: DECLS
            p_in, p_out = PRICES[settings["model"]]
            runs.append({"passed": score(out["answer"], case), "refused": refused(out["answer"]),
                         "answered": out["answer"] is not None, "answer": out["answer"], **model.tally,
                         "cost": (model.tally["in"] * p_in + model.tally["out"] * p_out) / 1e6})
        results[case["id"]] = runs
    return results

evals = {}
print("eval ready:", len(GOLDEN), "cases x 2 runs per model")

eval ready: 12 cases x 2 runs per model


### 6. Run the eval on the old model, then the new one
12 cases x 2 runs = 24 agent runs per model, same prompt, same tools, same scorer. Only the settings
dict differs. If these cells are skipped (no key, no recording), the cells after them say so.

In [14]:
evals["old"] = run_eval(BASELINE)
print("old:", {cid: "".join("P" if r["passed"] else "." for r in runs) for cid, runs in evals["old"].items()})

old: {'moons-count': 'PP', 'moon-names': 'PP', 'backwards': 'PP', 'largest': 'PP', 'rings': 'PP', 'rover-date': 'PP', 'day-length': 'PP', 'day-minutes': 'PP', 'day-seconds': 'PP', 'ten-days': 'PP', 'out-france': 'PP', 'out-neptune': 'PP'}


Now the candidate, on exactly the same 24 runs.

In [15]:
evals["new"] = run_eval(CANDIDATE)
print("new:", {cid: "".join("P" if r["passed"] else "." for r in runs) for cid, runs in evals["new"].items()})

new: {'moons-count': 'PP', 'moon-names': 'PP', 'backwards': 'PP', 'largest': 'PP', 'rings': 'PP', 'rover-date': 'PP', 'day-length': 'PP', 'day-minutes': 'PP', 'day-seconds': 'PP', 'ten-days': 'PP', 'out-france': 'PP', 'out-neptune': 'PP'}


### 7. The migration diff
One row per metric. **Wrong refusals** are refusals on questions the notes do answer; **missed
refusals** are answers to questions they do not. Both matter: a model that refuses more looks "safer"
and is less useful. The function refuses to compare runs on different cases (Common mistakes).

In [16]:
def metrics(results: dict) -> dict:
    inscope = [r for c in GOLDEN if not c.get("refuse") for r in results[c["id"]]]
    outscope = [r for c in GOLDEN if c.get("refuse") for r in results[c["id"]]]
    runs = inscope + outscope
    avg = lambda key: sum(r[key] for r in runs) / len(runs)
    return {"accuracy": sum(r["passed"] for r in runs) / len(runs),
            "wrong refusals": sum(r["refused"] for r in inscope), "missed refusals": sum(not r["refused"] for r in outscope),
            "no answer": sum(not r["answered"] for r in runs), "tool calls/run": avg("tool_calls"),
            "model calls/run": avg("calls"), "tokens in/run": avg("in"), "tokens out/run": avg("out"), "cost $/run": avg("cost")}

def diff(old: dict, new: dict) -> dict:
    if not old or not new:
        raise ValueError("an eval is missing: run both models first")
    if old.keys() != new.keys() or any(len(old[k]) != len(new[k]) for k in old):
        raise ValueError("old and new were run on different cases or run counts: the diff would be unfair")
    return {m: (a, metrics(new)[m]) for m, a in metrics(old).items()}

Print it, with the change as a ratio where that helps.

In [17]:
if {"old", "new"} <= evals.keys():
    table = diff(evals["old"], evals["new"])
    print(f"{'metric':<17}{'old':>10}{'new':>10}")
    for m, (a, b) in table.items():
        ratio = f"  x{b / a:.1f}" if a and m.endswith("/run") else ""
        print(f"{m:<17}{a:>10.4g}{b:>10.4g}{ratio}")
else:
    table = None
    print("skipped: run with a recording or a key to compare old and new")

metric                  old       new
accuracy                  1         1
wrong refusals            0         0
missed refusals           0         0
no answer                 0         0
tool calls/run        1.292     1.458  x1.1
model calls/run       2.292     2.417  x1.1
tokens in/run         353.1     387.1  x1.1
tokens out/run        40.71     60.75  x1.5
cost $/run        0.0002077 0.0005182  x2.5


Recorded 2026-10-09: both models passed all 24 runs, refusals included, so accuracy alone says "swap".
The other rows say more. The new model made slightly more tool calls (1.46 against 1.29 per run) and
wrote 50% more output tokens (its `LOW` thinking included). Its tokens also cost more (lesson 17's
prices), so each run costs 2.5 times as much.

Read answers side by side, as lesson 48 taught: the number says *that*, the text says *why*. Cases
whose verdict changed come first; when none changed, read a refusal and an arithmetic case anyway.

In [18]:
if table:
    changed = [c["id"] for c in GOLDEN if sum(r["passed"] for r in evals["old"][c["id"]])
               != sum(r["passed"] for r in evals["new"][c["id"]])]
    print("verdict changed on:", changed or "no case")
    for cid in changed or ["out-neptune", "ten-days"]:
        for v in ("old", "new"):
            print(f"{cid:<12}{v}: {(evals[v][cid][0]['answer'] or '(no answer)').strip()[:100]}")

verdict changed on: no case
out-neptune old: The notes do not contain information about how many moons Neptune has.
out-neptune new: Your notes do not contain information about how many moons Neptune has.
ten-days    old: Based on your notes stating that a day on Mars lasts about 24 hours and 39 minutes, 10 days on Mars 
ten-days    new: Based on your notes, a day on Mars lasts about 24 hours and 39 minutes, which means 10 days equals 2


### 8. Gates: block, review or go
Turn the diff into a decision you can run in CI. Quality metrics **block** if they get worse. Cost and
behaviour metrics ask for a human **review** if they move a lot, because a bigger bill may still be
worth it, but someone must say so.

In [19]:
def gate(table: dict, max_ratio: float = 1.5) -> tuple[str, list[str]]:
    block = [m for m in ("accuracy",) if table[m][1] < table[m][0]]
    block += [m for m in ("wrong refusals", "missed refusals", "no answer") if table[m][1] > table[m][0]]
    review = [f"{m} x{b / a:.1f}" for m, (a, b) in table.items()
              if m.endswith("/run") and a and not (1 / max_ratio <= b / a <= max_ratio)]
    if block:
        return "BLOCK", block
    return ("REVIEW", review) if review else ("GO", [])

print(gate(table) if table else "skipped: no diff")

('REVIEW', ['cost $/run x2.5'])


`REVIEW`: nothing got worse, but each run costs 2.5 times as much. At lesson 17's volume of 300,000
questions a month, that is about $93 more a month. A person decides if it is worth it: the newer model
may do better on harder questions this small golden set lacks, and Scout must leave flash-lite one day
anyway. Here the team accepts the cost, signs the review and starts a canary.

### 9. The canary
Passing the eval earns a canary, not a switch. A **canary** sends a small, fixed share of real users to
the new model while you watch the same metrics from logs (lesson 49). The split hashes the user id, so a
user stays on one model across requests instead of flipping between them.

In [20]:
MODELS = {"scout.answer": BASELINE, "scout.answer.canary": CANDIDATE}
CANARY_PERCENT = 10                      # 0 = rollback, 100 = done

def settings_for(user_id: str) -> dict:
    bucket = int(hashlib.sha256(user_id.encode()).hexdigest(), 16) % 100
    return MODELS["scout.answer.canary"] if bucket < CANARY_PERCENT else MODELS["scout.answer"]

users = [f"user-{i}" for i in range(1000)]
on_new = sum(settings_for(u)["model"] == CANDIDATE["model"] for u in users)
print(f"{on_new} of {len(users)} users on {CANDIDATE['model']}; user-7 twice:",
      settings_for("user-7")["model"], settings_for("user-7")["model"])

95 of 1000 users on gemini-3.8-flash; user-7 twice: gemini-3.5-flash-lite gemini-3.5-flash-lite


Rollback is one number: set `CANARY_PERCENT = 0`. Once the canary looks like the eval, move it to 100,
then make the candidate the new `scout.answer` and keep the old id in config until the vendor retires it.
Finally, the startup check from step 2 now covers both pins.

In [21]:
for role, s in MODELS.items():
    preflight(s)
    print(f"  {role:<21}{s['model']:<23}{lifecycle_status(s['model'], TODAY)}")

gemini-3.5-flash-lite settings pass preflight
  scout.answer         gemini-3.5-flash-lite  OK
gemini-3.8-flash settings pass preflight
  scout.answer.canary  gemini-3.8-flash       OK


## At work
- **Retirement is on the vendor's calendar, not yours.** `claude-haiku-4-5-20251001` retires "not sooner
  than 2026-10-15"; Gemini 2.5 is closed to new projects. Teams keep an inventory of every pinned id
  across services and subscribe to the deprecation pages. → lesson 56
- **New models change the bill as well as the answers.** Claude 4.7+ tokenizers produce about 30% more
  tokens for the same text, and thinking is billed as output, so cost per run must be in the diff.
  → lesson 54
- **A fallback model is a migration you haven't tested.** Switching to a second model in an outage hits
  the same rejected settings unless it passed preflight and the eval in advance. → lesson 57
- **The eval is the release gate.** Model swaps run the golden set in CI like prompt changes, and block
  on regressions. → lesson 55

## What just happened
- `LIFECYCLE` dated each model's end; `lifecycle_status` flagged a Haiku pin with 6 days left and a
  restricted Gemini 2.5 pin, and passed Scout's.
- A preflight call with Scout's production settings on `gemini-3.8-flash` got a real 400: the
  `MINIMAL` thinking level is not supported there. `preflight` encodes such rules, including Claude's
  rejected `temperature`.
- Per-model settings fixed it; the same 12-case eval ran on both models, recording tool calls, tokens,
  refusals and cost as well as pass/fail.
- Both models passed all 24 runs, but the new one cost 2.5 times as much per run, so `gate` returned
  `REVIEW`, not `GO`. `settings_for` sent 95 of 1,000 users to the canary by hash, with rollback as one
  number.

## Common mistakes
**1. Carrying sampling settings to a model that rejects them.** Scout's settings include
`temperature=1.0`. Moving them to Claude Sonnet 5.5 unchanged would 400 on every call; the preflight
check stops it before any request.

In [22]:
preflight({**BASELINE, "model": "claude-sonnet-5-5", "thinking_level": None})

ValueError: claude-sonnet-5-5: temperature is rejected; remove it

Fix: give each model its own settings, without the keys it rejects, and preflight them.

In [23]:
preflight({"model": "claude-sonnet-5-5", "max_output_tokens": 1024})

claude-sonnet-5-5 settings pass preflight


**2. Diffing evals run on different cases.** The new model was run on a "quick" eval without the two
out-of-scope questions. Its numbers look comparable, but it was never tested on refusals.

In [24]:
quick_new = {cid: runs for cid, runs in evals.get("new", {}).items() if not cid.startswith("out-")}
diff(evals.get("old", {}), quick_new)

ValueError: old and new were run on different cases or run counts: the diff would be unfair

Fix: run old and new on the same golden set with the same `n`, ideally in the same session, as step 6 did.

## Production checklist
- [ ] Every model id pinned in config, with a dated retirement entry and a source; checked at startup.
- [ ] Someone subscribed to each vendor's deprecation page; migrations start at least 30 days out.
- [ ] Settings stored per model; one preflight call with the full production settings before any eval.
- [ ] Old and new run on the same golden set, including out-of-scope cases, with the same `n`.
- [ ] The diff covers accuracy, wrong and missed refusals, missing answers, tool calls, tokens and cost.
- [ ] Quality regressions block; big cost or behaviour shifts need a named reviewer.
- [ ] Canary by hashed user id, rollback by setting the share to 0; old id kept until retired.

## Try it
Scout's team also wants the candidate at `MEDIUM` thinking. Without new model calls:
1. Add a `"gemini-3.8-flash-medium"` copy of `CANDIDATE` with `thinking_level="MEDIUM"` and check it with
   `preflight` (it uses the same `RULES` entry as `gemini-3.8-flash`: pass `model="gemini-3.8-flash"`).
2. Write `days_left(model_id, today)` that returns the days until retirement or `None`, and print it for
   every id in `LIFECYCLE` on 2026-10-09 and on 2027-09-01. Which ids need a migration plan by then?

In [25]:
# your code here

Finally, stop the recorder (in record mode this saves the recording).

In [26]:
llm_replay.stop()

## Key terms
- **deprecation** — a vendor's announcement that a model will stop working; the model still answers until its retirement date.
- **retirement** — the date after which calls to a model id fail; track it per pinned id.
- **model lifecycle table** — every pinned model id with its stage, retirement date and where that was read.
- **preflight call** — one cheap request with the full production settings to a new model, to catch rejected settings before an eval.
- **migration diff** — old vs new model on the same eval, comparing accuracy, refusals, tool calls, tokens and cost.
- **canary** — sending a small, fixed share of users to a new model before everyone, with rollback by setting the share to 0.